# 05 · Persiapan Data Meteorologi — Bab 6

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 6: memuat data, QC, eksplorasi, *feature engineering*, normalisasi, dan split berbasis waktu. Data contoh sintetik disediakan agar dapat dijalankan tanpa koneksi eksternal; ganti dengan data nyata (BMKG/ERA5) untuk proyek.

## 1. Setup & Data Contoh

Kita buat data harian sintetik yang meniru perilaku hujan: musiman + banyak hari nol + ekor panjang.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
t = pd.date_range("2010-01-01", "2021-12-31", freq="D")
n = len(t)

# musiman (monsun): kuat Nov-Mar
doy = t.dayofyear.values
musim = 15 + 18 * np.sin(2 * np.pi * (doy - 15) / 365.25) * (np.sin(2*np.pi*(doy-15)/365.25) > 0)
musim = np.maximum(musim, 0)

hujan = np.maximum(np.random.gamma(1.2, 6, n) + musim + 3 * np.random.randn(n), 0)
hujan = np.where(np.random.rand(n) < 0.55, 0.0, hujan)  # sebagian hari kering

suhu = 27 + 1.5 * np.sin(2 * np.pi * (doy - 60) / 365.25) + 1.0 * np.random.randn(n)
kelembapan = 75 + 8 * np.sin(2 * np.pi * (doy - 15) / 365.25) + 5 * np.random.randn(n)
df = pd.DataFrame({"r_hujan": hujan.round(1), "suhu": suhu.round(1),
                   "kelembapan": kelembapan.round(1)}, index=t)
df["r_hujan"][df.index.day == 29] = np.nan   # sisipkan gap contoh
print(df.head())
print(df.describe())

## 2. Quality Control & Imputasi

In [ ]:
print("Nilai hilang per kolom:")
print(df.isna().sum())

# imputasi sederhana: ffill untuk gap pendek
df_clean = df.copy()
df_clean["suhu"] = df_clean["suhu"].fillna(method="ffill")
df_clean["kelembapan"] = df_clean["kelembapan"].fillna(method="ffill")
df_clean["r_hujan"] = df_clean["r_hujan"].fillna(0.0)  # konservatif utk hujan
print("Sisa hilang:", int(df_clean.isna().sum().sum()))

## 3. Eksplorasi Distribusi (Gambar 6.1)

In [ ]:
plt.figure(figsize=(6.5,4))
df_clean["r_hujan"].hist(bins=60, color="#4a90e2", edgecolor="white")
plt.xlabel("Curah hujan harian (mm)"); plt.ylabel("Frekuensi")
plt.title("Distribusi curah hujan harian (contoh)")
plt.tight_layout(); plt.show()

## 4. Feature Engineering

Fitur: deret tunda, musiman sinus, dan (di sini) dummy indeks iklim.

In [ ]:
from sklearn.preprocessing import StandardScaler

df_feat = df_clean.copy()
for lag in [1, 2, 3, 7, 14]:
    df_feat[f"hujan_t{lag}"] = df_feat["r_hujan"].shift(lag)
    df_feat[f"suhu_t{lag}"] = df_feat["suhu"].shift(lag)

df_feat["mus_sin"] = np.sin(2 * np.pi * df_feat.index.dayofyear / 365.25)
df_feat["mus_cos"] = np.cos(2 * np.pi * df_feat.index.dayofyear / 365.25)

# dummy indeks iklim (ganti dengan data MJO/ENSO nyata pada proyek)
df_feat["rmm1"] = 0.8 * np.sin(2 * np.pi * np.arange(len(df_feat)) / 45.0)
df_feat["nino34"] = 0.5 + 1.2 * np.sin(2 * np.pi * np.arange(len(df_feat)) / 365.25 * 3)

feat_cols = [c for c in df_feat.columns if c != "r_hujan"]
print("Fitur:", feat_cols)

## 5. Housekeeping: buang baris awal (lag -> NaN) & split berbasis waktu

In [ ]:
df_ml = df_feat.dropna().copy()
data = df_ml[feat_cols]
target = df_ml["r_hujan"]

n = len(data)
n_train = int(n * 0.7); n_val = int(n * 0.15)

scale = StandardScaler().fit(data.iloc[:n_train])
X_train = scale.transform(data.iloc[:n_train])
X_val = scale.transform(data.iloc[n_train:n_train+n_val])
X_test = scale.transform(data.iloc[n_train+n_val:])
y_train, y_val, y_test = target.iloc[:n_train], target.iloc[n_train:n_train+n_val], target.iloc[n_train+n_val:]

print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("Rentang:", df_ml.index[0].date(), "->", df_ml.index[-1].date())

## 6. Latihan Mini

1. Ulangi dengan transformasi `log1p` pada target hujan; bandingkan distribusinya.
2. Buat fungsi pipeline reusable `make_dataset(csv_path) -> X, y, dates, scaler` untuk Bab 8–9.
3. Ganti dummy MJO/ENSO dengan data nyata (panduan tautan di Bab 6) dan periksa efektnya pada model.
4. Terapkan *walk-forward* sederhana dan bandingkan MAE dengan split tunggal.